# ☕ Ejemplo 2 — Intermedio: Coffee Bean Dataset con EfficientNetB0

**Curso:** IS056B — Ingeniería del Conocimiento · **Sesión 09-A: Transfer Learning — Feature Extraction**

**Basado en:** `Feature_Extraction_TL_Semana09_A.ipynb`, **Ejemplo 2** (TF Flowers), manteniendo sus etapas **2.1–2.6**, su arquitectura y las evaluaciones.

**Objetivo:** clasificar imágenes de granos de café según el nivel de tostado mediante **EfficientNetB0 preentrenado en ImageNet** (backbone congelado). Incluye EDA, Data Augmentation, entrenamiento, curvas, reporte de clasificación, matriz de confusión y feature maps.

**Dataset Kaggle:** [Coffee Bean Dataset Resized (224×224)](https://www.kaggle.com/datasets/gpiosenka/coffee-bean-dataset-resized-224-x-224) — **4 clases**: `Dark`, `Green`, `Light`, `Medium` (aprox. 1600 imágenes en total).

> **Nota:** Este notebook **no tiene métricas precalculadas**; los resultados se generan cuando ejecutes las celdas en Google Colab. Separamos validación y prueba para evitar usar el conjunto de prueba al ajustar el modelo.


## ⚙️ Configuración del entorno

1. Abre el notebook en [Google Colab](https://colab.research.google.com/).
2. Para acelerar el entrenamiento: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**, si está disponible.
3. Ejecuta las celdas **en orden**. La descarga pública de Kaggle se realiza con `kagglehub`. Si Kaggle exige autenticación, revisa sus instrucciones de acceso.


In [ ]:
# Librerías necesarias (Colab)
%pip -q install kagglehub scikit-learn seaborn


In [ ]:
# Verificar GPU e importar librerías
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
import kagglehub
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
BATCH_SIZE = 32
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 5)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'No disponible (se usará CPU)')


---
# 🟡 Ejemplo 2 — Intermedio: Coffee Bean Dataset con EfficientNetB0

**Objetivo:** Feature Extraction con **4 clases**, EDA detallado, Data Augmentation, métricas avanzadas (confusion matrix, classification report) y visualización de feature maps.

**Dataset:** [Coffee Bean Dataset Resized (224×224)](https://www.kaggle.com/datasets/gpiosenka/coffee-bean-dataset-resized-224-x-224), con clases `Dark`, `Green`, `Light` y `Medium`.

**¿Por qué EfficientNetB0?** Conservamos la arquitectura preentrenada del Ejemplo 2 de tu docente. Solo se entrena una nueva cabeza de clasificación para 4 clases.


### 2.1. Cargar dataset y EDA

In [ ]:
# Descargar Coffee Bean Dataset desde Kaggle
DATASET_SLUG = 'gpiosenka/coffee-bean-dataset-resized-224-x-224'
dataset_root = Path(kagglehub.dataset_download(DATASET_SLUG))
print('✅ Dataset descargado en:', dataset_root)
print('📁 Carpetas/archivos principales:', [p.name for p in dataset_root.iterdir()][:20])

# Localizar las cuatro clases, independientemente de si Kaggle organiza
# las imágenes en train/test, train/valid/test o en una sola carpeta.
EXPECTED = ['Dark', 'Green', 'Light', 'Medium']
EXTS = {'.png', '.jpg', '.jpeg', '.bmp'}

def buscar_raices_con_clases(base):
    raices = []
    for carpeta in [base, *base.rglob('*')]:
        if not carpeta.is_dir():
            continue
        subdirs = {h.name.lower(): h for h in carpeta.iterdir() if h.is_dir()}
        if all(clase.lower() in subdirs for clase in EXPECTED):
            raices.append(carpeta)
    return raices

roots = buscar_raices_con_clases(dataset_root)
if not roots:
    # Respaldo si la descarga contiene un ZIP interno.
    import zipfile
    archives = list(dataset_root.rglob('*.zip'))
    if archives:
        extracted = Path('/content/coffee_bean_extraido')
        extracted.mkdir(parents=True, exist_ok=True)
        for archive in archives:
            with zipfile.ZipFile(archive) as z:
                z.extractall(extracted)
        roots = buscar_raices_con_clases(extracted)

if not roots:
    raise FileNotFoundError('No se encontraron carpetas Dark, Green, Light y Medium. Revisa la estructura del dataset descargado.')

print('📂 Carpetas detectadas:')
for root in roots:
    print('  -', root)

CLASS_NAMES_2 = EXPECTED.copy()
NUM_CLASSES_2 = len(CLASS_NAMES_2)

def recopilar_imagenes(root):
    archivos, etiquetas = [], []
    child_by_name = {p.name.lower(): p for p in root.iterdir() if p.is_dir()}
    for idx, clase in enumerate(CLASS_NAMES_2):
        carpeta_clase = child_by_name[clase.lower()]
        imagenes = sorted(p for p in carpeta_clase.rglob('*') if p.is_file() and p.suffix.lower() in EXTS)
        archivos.extend(str(p) for p in imagenes)
        etiquetas.extend([idx] * len(imagenes))
    return archivos, etiquetas

def detectar_split(root):
    nombre = root.name.lower().strip().replace('_', ' ')
    if nombre in ('train', 'training', 'entrenamiento'):
        return 'train'
    if nombre in ('valid', 'validation', 'val', 'validacion'):
        return 'val'
    if nombre in ('test', 'testing', 'prueba'):
        return 'test'
    return 'other'

por_split = {}
for root in roots:
    split = detectar_split(root)
    if split in por_split:
        raise ValueError(f'Hay varias carpetas para el split {split}; revisa las rutas detectadas.')
    por_split[split] = recopilar_imagenes(root)

# Si el dataset proporciona train/test, reservamos parte del train para validación.
# Nunca convertimos el conjunto de test en validación para el entrenamiento.
if 'train' in por_split:
    train_paths, train_labels = por_split['train']
    if 'val' in por_split:
        val_paths, val_labels = por_split['val']
    else:
        train_paths, val_paths, train_labels, val_labels = train_test_split(
            train_paths, train_labels, test_size=0.2, random_state=SEED, stratify=train_labels
        )
    if 'test' in por_split:
        test_paths, test_labels = por_split['test']
    else:
        # Solo cuando no hay test oficial, separar una parte del entrenamiento.
        train_paths, test_paths, train_labels, test_labels = train_test_split(
            train_paths, train_labels, test_size=0.15, random_state=SEED, stratify=train_labels
        )
elif len(roots) == 1:
    all_paths, all_labels = recopilar_imagenes(roots[0])
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        all_paths, all_labels, test_size=0.30, random_state=SEED, stratify=all_labels)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.50, random_state=SEED, stratify=temp_labels)
else:
    raise ValueError('La disposición de carpetas es inusual. Revisa las rutas impresas arriba.')

assert all(len(x) for x in (train_paths, val_paths, test_paths)), 'Hay un split vacío.'
assert set(train_paths).isdisjoint(val_paths) and set(train_paths).isdisjoint(test_paths) and set(val_paths).isdisjoint(test_paths)

# tf.data: conservamos los nombres de las variables del Ejemplo 2 original.
def cargar_imagen(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape((None, None, 3))
    return image, label

def desde_archivos(paths, labels):
    return tf.data.Dataset.from_tensor_slices((paths, np.asarray(labels, dtype=np.int32))).map(
        cargar_imagen, num_parallel_calls=tf.data.AUTOTUNE)

train_raw_2 = desde_archivos(train_paths, train_labels)
val_raw_2 = desde_archivos(val_paths, val_labels)
test_raw_2 = desde_archivos(test_paths, test_labels)

print(f'☕ Clases ({NUM_CLASSES_2}): {CLASS_NAMES_2}')
print(f'📊 Train: {len(train_paths):,} | Val: {len(val_paths):,} | Test: {len(test_paths):,}')
print('📊 Total:', len(train_paths) + len(val_paths) + len(test_paths))


In [ ]:
# Distribución de clases
train_labels = [label.numpy() for _, label in train_raw_2]
val_labels = [label.numpy() for _, label in val_raw_2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, labels, title_str in [(axes[0], train_labels, 'Train'),
                               (axes[1], val_labels, 'Validation')]:
    unique, counts = np.unique(labels, return_counts=True)
    colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES_2))
    bars = ax.bar([CLASS_NAMES_2[u] for u in unique], counts, color=colors, edgecolor='gray')
    ax.set_title(f'Distribución — {title_str}', fontweight='bold')
    ax.set_ylabel('Cantidad')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
               str(count), ha='center', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Coffee Beans — Distribución de clases', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Muestras por clase
fig, axes = plt.subplots(NUM_CLASSES_2, 4, figsize=(14, 3 * NUM_CLASSES_2))
class_iterators = {i: iter(train_raw_2.filter(lambda x, y: y == i)) for i in range(NUM_CLASSES_2)}

for class_idx in range(NUM_CLASSES_2):
    for col in range(4):
        ax = axes[class_idx, col]
        try:
            img, _ = next(class_iterators[class_idx])
            ax.imshow(img.numpy())
        except StopIteration:
            pass
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(CLASS_NAMES_2[class_idx], fontsize=12, fontweight='bold',
                         rotation=0, labelpad=60, va='center')

plt.suptitle('Muestras por clase — Coffee Beans', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 2.2. Preprocesamiento con Data Augmentation

Para mejorar la generalización, aplicamos **Data Augmentation solamente al entrenamiento**, como en el ejemplo original: giros, rotaciones, zoom y contraste.

**Importante:** En `tf.keras.applications.EfficientNetB0`, el preprocesamiento de píxeles ya está integrado en el modelo; `efficientnet.preprocess_input` es una operación identidad. Se conservan imágenes en rango **[0, 255]** (no normalizarlas otra vez a [0,1]). La clase de tostado depende del color, por eso conviene evaluar si demasiado aumento de contraste perjudica el resultado.


In [ ]:
IMG_SIZE_2 = (224, 224)

# Capa de augmentation (solo se aplica en entrenamiento)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name='augmentation')

def preprocess_train_2(image, label):
    image = tf.image.resize(image, IMG_SIZE_2)
    image = tf.cast(image, tf.float32)
    image = data_augmentation(image, training=True)
    image = tf.keras.applications.efficientnet.preprocess_input(image)
    label = tf.one_hot(label, NUM_CLASSES_2)
    return image, label

def preprocess_val_2(image, label):
    image = tf.image.resize(image, IMG_SIZE_2)
    image = tf.keras.applications.efficientnet.preprocess_input(
        tf.cast(image, tf.float32))
    label = tf.one_hot(label, NUM_CLASSES_2)
    return image, label

# Shuffling antes del mapa para obtener nuevas variaciones aleatorias en cada época.
train_ds_2 = (train_raw_2
    .shuffle(min(1000, len(train_paths)), seed=SEED)
    .map(preprocess_train_2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

val_ds_2 = (val_raw_2
    .map(preprocess_val_2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

test_ds_2 = (test_raw_2
    .map(preprocess_val_2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

print("✅ Pipelines listos: augmentation SOLO en train")

In [ ]:
# Visualizar augmentation
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
sample_img, sample_label = next(iter(train_raw_2))
sample_resized = tf.image.resize(tf.cast(sample_img, tf.float32), IMG_SIZE_2)

axes[0, 0].imshow(sample_img.numpy())
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')

for i in range(1, 10):
    ax = axes[i // 5, i % 5]
    augmented = data_augmentation(tf.expand_dims(sample_resized, 0), training=True)
    ax.imshow(augmented[0].numpy().astype('uint8'))
    ax.set_title(f'Augmented #{i}', fontsize=10)
    ax.axis('off')

plt.suptitle(f'Data Augmentation — {CLASS_NAMES_2[sample_label.numpy()]}',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3. Construir modelo con EfficientNetB0

In [ ]:
# Backbone EfficientNetB0
base_model_2 = tf.keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMG_SIZE_2, 3)
)
base_model_2.trainable = False  # Congelar

# Modelo con clasificador más robusto
inputs_2 = layers.Input(shape=(*IMG_SIZE_2, 3))
x = base_model_2(inputs_2, training=False)  # BN en modo inferencia
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs_2 = layers.Dense(NUM_CLASSES_2, activation='softmax')(x)

model_2 = models.Model(inputs_2, outputs_2, name='coffee_efficientnetb0_FE')

# Resumen
total = model_2.count_params()
trainable = sum(tf.size(w).numpy() for w in model_2.trainable_weights)
print(f"📐 EfficientNetB0 — Feature Extraction")
print(f"🔢 Total: {total:,} | 🔓 Entrenables: {trainable:,} | 🔒 Congelados: {total-trainable:,}")

### 2.4. Entrenar con callbacks profesionales

In [ ]:
model_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_2 = model_2.fit(
    train_ds_2,
    validation_data=val_ds_2,
    epochs=15,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ]
)

print(f"\n🏆 Mejor val_accuracy: {max(history_2.history['val_accuracy']):.4f}")

In [ ]:
# Curvas de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ep = range(1, len(history_2.history['accuracy']) + 1)
axes[0].plot(ep, history_2.history['accuracy'], 'o-', label='Train', color='#4CAF50')
axes[0].plot(ep, history_2.history['val_accuracy'], 's-', label='Val', color='#E91E63')
axes[0].set_title('Accuracy — Coffee Beans (EfficientNetB0 FE)', fontweight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history_2.history['loss'], 'o-', label='Train', color='#4CAF50')
axes[1].plot(ep, history_2.history['val_loss'], 's-', label='Val', color='#E91E63')
axes[1].set_title('Loss — Coffee Beans (EfficientNetB0 FE)', fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.5. Evaluación detallada: Confusion Matrix + Classification Report

Conservamos las métricas del ejercicio original y añadimos un **conjunto de prueba independiente**. Las predicciones y la matriz de confusión se calculan sobre **test**, no sobre las imágenes usadas para optimizar los callbacks.


In [ ]:
# Evaluación cuantitativa sobre validación y prueba
val_loss_2, val_acc_2 = model_2.evaluate(val_ds_2, verbose=0)
test_loss_2, test_acc_2 = model_2.evaluate(test_ds_2, verbose=0)
print(f'✅ Validación — loss: {val_loss_2:.4f}, accuracy: {val_acc_2:.4f}')
print(f'✅ Prueba     — loss: {test_loss_2:.4f}, accuracy: {test_acc_2:.4f}')

# Classification Report (mismo procedimiento del Ejemplo 2, ahora sobre test)
y_true, y_pred = [], []
for images, labels in test_ds_2:
    preds = model_2.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print('\n📊 Classification Report — Coffee Beans (EfficientNetB0 FE)')
print('=' * 70)
print(classification_report(y_true, y_pred,
                            labels=np.arange(NUM_CLASSES_2),
                            target_names=CLASS_NAMES_2,
                            digits=4, zero_division=0))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES_2))
fig, ax = plt.subplots(figsize=(8, 7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=CLASS_NAMES_2, yticklabels=CLASS_NAMES_2, ax=ax,
           linewidths=0.5, linecolor='white',
           annot_kws={'size': 14, 'fontweight': 'bold'})

ax.set_xlabel('Predicción', fontsize=13, fontweight='bold')
ax.set_ylabel('Real', fontsize=13, fontweight='bold')
ax.set_title('Confusion Matrix — Coffee Beans (EfficientNetB0 FE)',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top errores
print("\n🔍 Principales confusiones:")
for i in range(NUM_CLASSES_2):
    for j in range(NUM_CLASSES_2):
        if i != j and cm[i, j] > 2:
            print(f"   {CLASS_NAMES_2[i]} → {CLASS_NAMES_2[j]}: {cm[i, j]} errores")

### 2.6. Visualización de Feature Maps

¿Qué características de forma, textura y brillo detecta EfficientNetB0 en los granos de café? Visualizamos activaciones del backbone congelado.


In [ ]:
# Seleccionar capas para visualizar
layer_names = ['block2a_expand_conv', 'block4a_expand_conv', 'block6a_expand_conv', 'top_conv']
feature_models = [
    models.Model(inputs=base_model_2.input,
                outputs=base_model_2.get_layer(name).output)
    for name in layer_names
]

# Tomar una imagen de ejemplo
sample_img_raw, sample_label_raw = next(iter(test_raw_2))
sample_processed = tf.image.resize(tf.cast(sample_img_raw, tf.float32), IMG_SIZE_2)
sample_processed = tf.keras.applications.efficientnet.preprocess_input(
    tf.expand_dims(sample_processed, 0))

# Extraer features
fig, axes = plt.subplots(len(layer_names), 9, figsize=(18, 2.5 * len(layer_names)))

for row, (feat_model, layer_name) in enumerate(zip(feature_models, layer_names)):
    features = feat_model.predict(sample_processed, verbose=0)
    # Mostrar la imagen original en la primera columna
    axes[row, 0].imshow(sample_img_raw.numpy())
    axes[row, 0].set_title('Original' if row == 0 else '', fontsize=9)
    axes[row, 0].set_ylabel(layer_name.replace('_', '\n'), fontsize=9,
                           fontweight='bold', rotation=0, labelpad=60, va='center')
    axes[row, 0].axis('off')
    # Mostrar 8 feature maps
    for col in range(1, 9):
        ax = axes[row, col]
        if col - 1 < features.shape[-1]:
            ax.imshow(features[0, :, :, col - 1], cmap='viridis')
        ax.axis('off')
        if row == 0:
            ax.set_title(f'Ch {col}', fontsize=9)

plt.suptitle(f'Feature Maps por Capa — {CLASS_NAMES_2[sample_label_raw.numpy()]}',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> 📝 **Observa la progresión**: las capas iniciales suelen responder a bordes y texturas, las intermedias a patrones más complejos de la superficie del grano, y las profundas a representaciones abstractas. Son los features transferidos desde ImageNet. La interpretación exacta de un canal requiere análisis adicional.


### ✅ Conclusión Ejemplo 2

| Elemento | Adaptación al café |
|---|---|
| **Modelo base** | EfficientNetB0 preentrenado en ImageNet, congelado |
| **Salida** | 4 neuronas con activación `softmax` |
| **Clases** | Dark, Green, Light, Medium |
| **Augmentation** | Flip + Rotation + Zoom + Contrast, solo en entrenamiento |
| **Entrenamiento** | Adam, `categorical_crossentropy`, hasta 15 épocas |
| **Callbacks** | EarlyStopping + ReduceLROnPlateau |
| **Evaluación** | Curvas, accuracy/loss, Classification Report y Confusion Matrix |
| **Visualización** | Feature maps de 4 capas del backbone |

> Las métricas exactas de **accuracy, precision, recall y F1-score** se observan al ejecutar el notebook. No se asumen cifras antes de entrenar.


---
## ✨ Complemento opcional — Predicciones visuales y guardar el modelo

Esta parte no cambia los pasos del Ejemplo 2 original. Permite demostrar las predicciones en tu exposición y conservar el modelo entrenado.


In [ ]:
# Predicciones sobre 8 imágenes del conjunto de prueba
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (img, etiqueta) in zip(axes.ravel(), test_raw_2.take(8)):
    entrada = tf.image.resize(tf.cast(img, tf.float32), IMG_SIZE_2)
    entrada = tf.expand_dims(entrada, axis=0)  # [1, 224, 224, 3]
    probs = model_2.predict(entrada, verbose=0)[0]
    idx = int(np.argmax(probs))
    real = CLASS_NAMES_2[int(etiqueta.numpy())]
    pred = CLASS_NAMES_2[idx]
    ax.imshow(img.numpy())
    ax.set_title(f'Real: {real}\nPred: {pred} ({probs[idx]:.1%})',
                 color='green' if real == pred else 'crimson', fontsize=10)
    ax.axis('off')
plt.suptitle('Predicciones de tostado de café — imágenes nunca usadas para entrenar', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Guardar el modelo y el orden exacto de clases
from pathlib import Path
import json
output_dir = Path('/content') if Path('/content').exists() else Path.cwd()
model_path = output_dir / 'coffee_efficientnetb0_FE.keras'
labels_path = output_dir / 'coffee_clases.json'
model_2.save(str(model_path))
labels_path.write_text(json.dumps(CLASS_NAMES_2, indent=2), encoding='utf-8')
print('✅ Modelo guardado:', model_path)
print('✅ Clases guardadas:', labels_path)
# En Colab, descarga manualmente los archivos desde el panel lateral Archivos.
